In [ ]:
import os
import requests
from requests_oauthlib import OAuth1
from dotenv import load_dotenv

# Load credentials from .env
load_dotenv()

ACCOUNT_ID   = os.getenv("NS_ACCOUNT_ID")
CONSUMER_KEY = os.getenv("NS_CONSUMER_KEY")
CONSUMER_SEC = os.getenv("NS_CONSUMER_SECRET")
TOKEN_ID     = os.getenv("NS_TOKEN_ID")
TOKEN_SEC    = os.getenv("NS_TOKEN_SECRET")

# Build the OAuth1 auth object
auth = OAuth1(
    CONSUMER_KEY,
    CONSUMER_SEC,
    TOKEN_ID,
    TOKEN_SEC,
    signature_method="HMAC-SHA256",
    realm=ACCOUNT_ID
)

# NetSuite SuiteQL endpoint
BASE_URL = f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/query/v1/suiteql"

def run_suiteql(query: str, limit: int = 100, offset: int = 0):
    """Run a SuiteQL query and return results as a list of dicts."""
    payload = {"q": query}
    headers = {"prefer": "transient", "Content-Type": "application/json"}

    response = requests.post(
        BASE_URL,
        auth=auth,
        json=payload,
        headers=headers,
        params={"limit": limit, "offset": offset}
    )
    response.raise_for_status()
    return response.json().get("items", [])

# --- Test Query ---
results = run_suiteql("SELECT id, companyName, email FROM customer WHERE rownum <= 10")

for row in results:
    print(row)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.head()